In [ ]:
# ============================================================
# Cell 1: Setup — clone repo to local disk, mount Drive for saving results
# ============================================================
import os

# Clone project repo to local disk (data + utils all included)
if not os.path.exists('/content/Project2'):
    !git clone https://github.com/sophie99zyq/NTU-AdvancedCV-Project2.git /content/Project2

# CycleGAN repo
if not os.path.exists('pytorch-CycleGAN-and-pix2pix'):
    !git clone https://github.com/junyanz/pytorch-CycleGAN-and-pix2pix.git
    !pip install -q visdom dominate

# Mount Drive for saving results only
from google.colab import drive
drive.mount('/content/drive')

import sys
sys.path.insert(0, '/content/Project2')

import torch
import torch.utils.data as data_utils
from torch.utils.data import TensorDataset, DataLoader
from torchvision.utils import save_image
from torchvision import datasets, transforms
from PIL import Image
import glob
import shutil
from utils.data_utils import get_office_home, get_paired_loaders, get_unnormalized_transform
from utils.fft_utils import fda_transfer
from utils.classifiers import ResNet50Classifier, train_classifier, evaluate_classifier
from utils.viz_utils import show_image_grid, print_results_table
from utils.eval_utils import save_results
from utils.cyclegan_wrapper import get_cyclegan_train_cmd, get_cyclegan_test_cmd, prepare_cyclegan_data
from utils.spectral_cyclegan import save_low_freq_images, reconstruct_from_translated_low
from utils.cycada import train_cycada, translate_with_cycada

device = 'cuda' if torch.cuda.is_available() else 'cpu'
DATA_ROOT = '/content/Project2/data/office_home'
SAVE_DIR = '/content/drive/MyDrive/Project2/results/art_realworld'
os.makedirs(SAVE_DIR, exist_ok=True)

print(f'Device: {device}')
print(f'Data: {DATA_ROOT}')
print(f'Save dir: {SAVE_DIR}')

In [ ]:
# === Quick Test Mode ===
# Set FAST_TEST = True to verify the notebook runs end-to-end (~5 min)
# Set FAST_TEST = False for full experiment
FAST_TEST = True

if FAST_TEST:
    CYCLEGAN_EPOCHS = 1
    CYCLEGAN_DECAY = 0
    CYCADA_EPOCHS = 1
    CLASSIFIER_EPOCHS = 2
    MAX_IMAGES = 100
    NUM_TEST = 50
else:
    CYCLEGAN_EPOCHS = 50
    CYCLEGAN_DECAY = 50
    CYCADA_EPOCHS = 50
    CLASSIFIER_EPOCHS = 20
    MAX_IMAGES = 5000
    NUM_TEST = 60000

print(f'FAST_TEST={FAST_TEST}: epochs={CYCLEGAN_EPOCHS}+{CYCLEGAN_DECAY}, images={MAX_IMAGES}')

In [ ]:
# ============================================================
# Cell 2: Load Data
# ============================================================
# For classifier (with ImageNet normalization)
art_train = get_office_home(root=DATA_ROOT, domain='Art', img_size=224)
realworld_train = get_office_home(root=DATA_ROOT, domain='Real World', img_size=224)

source_loader, target_loader = get_paired_loaders(art_train, realworld_train, batch_size=32)
target_test_loader = DataLoader(realworld_train, batch_size=32, shuffle=False)

# For CycleGAN/FDA (without normalization) - need unnormalized versions
unnorm_transform = get_unnormalized_transform(img_size=224)
art_unnorm = datasets.ImageFolder(os.path.join(DATA_ROOT, 'Art'), transform=unnorm_transform)
realworld_unnorm = datasets.ImageFolder(os.path.join(DATA_ROOT, 'Real World'), transform=unnorm_transform)
source_loader_unnorm, target_loader_unnorm = get_paired_loaders(art_unnorm, realworld_unnorm, batch_size=32)

print(f"Art: {len(art_train)}, Real World: {len(realworld_train)}")

# Task I: Style Transfer Comparison

In [ ]:
# ============================================================
# Cell 4: Save unnormalized images for CycleGAN
# ============================================================
def save_dataset_images(dataset, save_dir, max_images=None):
    """Save dataset images as individual PNG files for CycleGAN."""
    os.makedirs(save_dir, exist_ok=True)
    n = len(dataset) if max_images is None else min(max_images, len(dataset))
    for i in range(n):
        img, _ = dataset[i]
        save_image(img, os.path.join(save_dir, f'{i:06d}.png'))
    print(f'Saved {n} images to {save_dir}')

save_dataset_images(art_unnorm, '/content/cyclegan_data/art2realworld/trainA', max_images=MAX_IMAGES)
save_dataset_images(realworld_unnorm, '/content/cyclegan_data/art2realworld/trainB', max_images=MAX_IMAGES)

In [ ]:
# ============================================================
# Cell 5: Train Standard CycleGAN (spatial)
# ============================================================
cmd = get_cyclegan_train_cmd(
    n_epochs=CYCLEGAN_EPOCHS,
    n_epochs_decay=CYCLEGAN_DECAY,
    dataroot='/content/cyclegan_data/art2realworld',
    name='art2realworld_spatial',
    load_size=256,
    crop_size=224,
    input_nc=3,
    output_nc=3,
    netG='resnet_9blocks',
)
print(f'Running: {cmd}')
!{cmd}

In [ ]:
# ============================================================
# Cell 6: Train Spectral CycleGAN
# ============================================================
# Decompose source and target images into low/high frequency
src_low_dir, src_high_dir = save_low_freq_images(
    '/content/cyclegan_data/art2realworld/trainA',
    '/content/cyclegan_data/art2realworld_spectral/source_decomposed',
    beta=0.03,
)
tgt_low_dir, tgt_high_dir = save_low_freq_images(
    '/content/cyclegan_data/art2realworld/trainB',
    '/content/cyclegan_data/art2realworld_spectral/target_decomposed',
    beta=0.03,
)

# Prepare low-freq images for CycleGAN
prepare_cyclegan_data(src_low_dir, tgt_low_dir, '/content/cyclegan_data/art2realworld_spectral_lowfreq')

# Train CycleGAN on low-freq only
cmd_spectral = get_cyclegan_train_cmd(
    n_epochs=CYCLEGAN_EPOCHS,
    n_epochs_decay=CYCLEGAN_DECAY,
    dataroot='/content/cyclegan_data/art2realworld_spectral_lowfreq',
    name='art2realworld_spectral',
    load_size=256,
    crop_size=224,
    input_nc=3,
    output_nc=3,
    netG='resnet_9blocks',
)
print(f'Running: {cmd_spectral}')
!{cmd_spectral}

In [ ]:
for src, dst in [
    ('art2realworld/trainA', 'art2realworld/testA'),
    ('art2realworld/trainB', 'art2realworld/testB'),
    ('art2realworld_spectral_lowfreq/trainA', 'art2realworld_spectral_lowfreq/testA'),
    ('art2realworld_spectral_lowfreq/trainB', 'art2realworld_spectral_lowfreq/testB'),
]:
    src_dir = f'/content/cyclegan_data/{src}'
    dst_dir = f'/content/cyclegan_data/{dst}'
    os.makedirs(dst_dir, exist_ok=True)
    for f in sorted(glob.glob(f'{src_dir}/*.png'))[:50]:
        shutil.copy2(f, dst_dir)
    print(f"Copied {len(os.listdir(dst_dir))} images to {dst_dir}")

In [ ]:
# ============================================================
# Cell 7: Task I Visualization
# ============================================================
# Run CycleGAN inference (spatial)
cmd_test_spatial = get_cyclegan_test_cmd(
    netG='resnet_9blocks',
    dataroot='/content/cyclegan_data/art2realworld',
    name='art2realworld_spatial',
    load_size=256,
    crop_size=224,
    input_nc=3,
    output_nc=3,
)
!{cmd_test_spatial}

# Run CycleGAN inference (spectral low-freq)
cmd_test_spectral = get_cyclegan_test_cmd(
    netG='resnet_9blocks',
    dataroot='/content/cyclegan_data/art2realworld_spectral_lowfreq',
    name='art2realworld_spectral',
    load_size=256,
    crop_size=224,
    input_nc=3,
    output_nc=3,
)
!{cmd_test_spectral}

# Load and visualize results
to_tensor = transforms.ToTensor()
n_show = 8

# Original Art images (unnormalized for display)
original_imgs = torch.stack([art_unnorm[i][0] for i in range(n_show)])

# Spatial CycleGAN translated
spatial_dir = 'results/art2realworld_spatial/test_latest/images'
spatial_files = sorted(glob.glob(os.path.join(spatial_dir, '*_fake_B.png')))[:n_show]
spatial_imgs = torch.stack([to_tensor(Image.open(f).convert('RGB')) for f in spatial_files])

# Spectral CycleGAN: reconstruct translated low-freq + original high-freq
spectral_dir = 'results/art2realworld_spectral/test_latest/images'
spectral_files = sorted(glob.glob(os.path.join(spectral_dir, '*_fake_B.png')))[:n_show]
if len(spectral_files) > 0:
    spectral_low_imgs = torch.stack([to_tensor(Image.open(f).convert('RGB')) for f in spectral_files])
    high_freq_files = sorted(os.listdir(src_high_dir))[:n_show]
    high_imgs = torch.stack([to_tensor(Image.open(os.path.join(src_high_dir, f)).convert('RGB')) for f in high_freq_files])
    from utils.fft_utils import spectral_reconstruct
    spectral_imgs = spectral_reconstruct(spectral_low_imgs, high_imgs)
else:
    spectral_imgs = original_imgs  # fallback

# Target Real World images for reference
target_imgs = torch.stack([realworld_unnorm[i][0] for i in range(n_show)])

show_image_grid(
    {
        'Original Art': original_imgs,
        'Spatial CycleGAN': spatial_imgs,
        'Spectral CycleGAN': spectral_imgs,
        'Target Real World': target_imgs,
    },
    nrow=n_show,
    title='Task I: Art -> Real World Style Transfer Comparison',
    save_path=os.path.join(SAVE_DIR, 'task1_style_transfer.png'),
)

# Task II: UDA Benchmark

In [ ]:
# ============================================================
# Cell 9: Source-Only Baseline
# ============================================================
results = {}

# Train ResNet50 on Art (source only), test on Real World
model_source = ResNet50Classifier(num_classes=65).to(device)
model_source = train_classifier(model_source, source_loader, epochs=CLASSIFIER_EPOCHS, lr=1e-4, device=device)

acc_source_only = evaluate_classifier(model_source, target_test_loader, device=device)
results['Source Only'] = acc_source_only

In [ ]:
# ============================================================
# Cell 10: CycleGAN UDA
# ============================================================
# CycleGAN outputs are unnormalized [0,1] images.
# The classifier expects ImageNet-normalized inputs.
normalize = transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])

# Build dataset from spatial CycleGAN translated images
class TranslatedDataset(torch.utils.data.Dataset):
    def __init__(self, image_dir, original_dataset, suffix='_fake_B.png'):
        self.image_dir = image_dir
        self.files = sorted(glob.glob(os.path.join(image_dir, f'*{suffix}')))
        self.original_dataset = original_dataset
        self.transform = transforms.Compose([
            transforms.Resize((224, 224)),
            transforms.ToTensor(),
        ])
        self.normalize = transforms.Normalize(
            mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]
        )

    def __len__(self):
        return len(self.files)

    def __getitem__(self, idx):
        img = Image.open(self.files[idx]).convert('RGB')
        img = self.transform(img)
        img = self.normalize(img)
        _, label = self.original_dataset[idx]
        return img, label

translated_dir = 'results/art2realworld_spatial/test_latest/images'
cyclegan_dataset = TranslatedDataset(translated_dir, art_unnorm)
cyclegan_loader = DataLoader(cyclegan_dataset, batch_size=32, shuffle=True)

model_cyclegan = ResNet50Classifier(num_classes=65).to(device)
model_cyclegan = train_classifier(model_cyclegan, cyclegan_loader, epochs=CLASSIFIER_EPOCHS, lr=1e-4, device=device)
acc_cyclegan = evaluate_classifier(model_cyclegan, target_test_loader, device=device)
results['CycleGAN'] = acc_cyclegan

In [ ]:
# ============================================================
# Cell 11: Spectral CycleGAN UDA
# ============================================================
# Reconstruct full images from translated low-freq + original high-freq
spectral_translated_dir = 'results/art2realworld_spectral/test_latest/images'
spectral_recon_dir = '/content/cyclegan_data/art2realworld_spectral_reconstructed'

# Copy translated low-freq fake_B images to a temp dir with matching filenames
translated_low_files = sorted(glob.glob(os.path.join(spectral_translated_dir, '*_fake_B.png')))
translated_low_dir_tmp = '/content/cyclegan_data/art2realworld_spectral_translated_low'
os.makedirs(translated_low_dir_tmp, exist_ok=True)
for i, f in enumerate(translated_low_files):
    shutil.copy(f, os.path.join(translated_low_dir_tmp, f'{i:06d}.png'))

# Reconstruct: translated low-freq + original high-freq
reconstruct_from_translated_low(translated_low_dir_tmp, src_high_dir, spectral_recon_dir)

# Build dataset from reconstructed images with ImageNet normalization
class ReconstructedDataset(torch.utils.data.Dataset):
    def __init__(self, image_dir, original_dataset):
        self.image_dir = image_dir
        self.files = sorted([f for f in os.listdir(image_dir) if f.endswith('.png')])
        self.original_dataset = original_dataset
        self.transform = transforms.Compose([
            transforms.Resize((224, 224)),
            transforms.ToTensor(),
        ])
        self.normalize = transforms.Normalize(
            mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]
        )

    def __len__(self):
        return len(self.files)

    def __getitem__(self, idx):
        img = Image.open(os.path.join(self.image_dir, self.files[idx])).convert('RGB')
        img = self.transform(img)
        img = self.normalize(img)
        _, label = self.original_dataset[idx]
        return img, label

spectral_dataset = ReconstructedDataset(spectral_recon_dir, art_unnorm)
spectral_loader = DataLoader(spectral_dataset, batch_size=32, shuffle=True)

model_spectral = ResNet50Classifier(num_classes=65).to(device)
model_spectral = train_classifier(model_spectral, spectral_loader, epochs=CLASSIFIER_EPOCHS, lr=1e-4, device=device)
acc_spectral = evaluate_classifier(model_spectral, target_test_loader, device=device)
results['Spectral CycleGAN'] = acc_spectral

In [ ]:
# ============================================================
# Cell 12: CyCADA
# ============================================================
# CyCADA works on unnormalized [0,1] images internally.
# We need a source-only classifier trained on unnormalized images for the task loss.

model_source_unnorm = ResNet50Classifier(num_classes=65).to(device)
model_source_unnorm = train_classifier(
    model_source_unnorm, source_loader_unnorm, epochs=CLASSIFIER_EPOCHS, lr=1e-4, device=device
)

G_s2t, G_t2s = train_cycada(
    source_loader_unnorm, target_loader_unnorm, model_source_unnorm,
    num_classes=65, in_channels=3,
    n_epochs=CYCADA_EPOCHS, lr=2e-4,
    lambda_cyc=10.0, lambda_task=1.0,
    device=device,
)

# Translate source images to target domain (outputs [0,1] images)
translated_data = translate_with_cycada(G_s2t, source_loader_unnorm, device=device)

# Create DataLoader with ImageNet normalization applied to translated images
normalize = transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])

class CycadaDataset(torch.utils.data.Dataset):
    def __init__(self, translated_data, normalize_fn):
        self.images = torch.cat([imgs for imgs, _ in translated_data], dim=0)
        self.labels = torch.cat([labels for _, labels in translated_data], dim=0)
        self.normalize_fn = normalize_fn

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        img = self.normalize_fn(self.images[idx])
        return img, self.labels[idx]

cycada_dataset = CycadaDataset(translated_data, normalize)
cycada_loader = DataLoader(cycada_dataset, batch_size=32, shuffle=True)

model_cycada = ResNet50Classifier(num_classes=65).to(device)
model_cycada = train_classifier(model_cycada, cycada_loader, epochs=CLASSIFIER_EPOCHS, lr=1e-4, device=device)
acc_cycada = evaluate_classifier(model_cycada, target_test_loader, device=device)
results['CyCADA'] = acc_cycada

In [ ]:
# ============================================================
# Cell 13: FDA (Fourier Domain Adaptation)
# ============================================================
# FDA operates on unnormalized [0,1] images, then we normalize for the classifier.
normalize = transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])

class FDADataset(torch.utils.data.Dataset):
    def __init__(self, source_loader, target_loader, beta=0.01, normalize_fn=None):
        self.source_images = []
        self.source_labels = []
        self.normalize_fn = normalize_fn
        target_iter = iter(target_loader)
        for src_imgs, src_labels in source_loader:
            try:
                tgt_imgs, _ = next(target_iter)
            except StopIteration:
                target_iter = iter(target_loader)
                tgt_imgs, _ = next(target_iter)
            bs = min(src_imgs.size(0), tgt_imgs.size(0))
            transferred = fda_transfer(src_imgs[:bs], tgt_imgs[:bs], beta=beta)
            self.source_images.append(transferred)
            self.source_labels.append(src_labels[:bs])
        self.source_images = torch.cat(self.source_images, dim=0)
        self.source_labels = torch.cat(self.source_labels, dim=0)

    def __len__(self):
        return len(self.source_labels)

    def __getitem__(self, idx):
        img = self.source_images[idx]
        if self.normalize_fn is not None:
            img = self.normalize_fn(img)
        return img, self.source_labels[idx]

fda_dataset = FDADataset(
    source_loader_unnorm, target_loader_unnorm, beta=0.01, normalize_fn=normalize
)
fda_loader = DataLoader(fda_dataset, batch_size=32, shuffle=True)

model_fda = ResNet50Classifier(num_classes=65).to(device)
model_fda = train_classifier(model_fda, fda_loader, epochs=CLASSIFIER_EPOCHS, lr=1e-4, device=device)
acc_fda = evaluate_classifier(model_fda, target_test_loader, device=device)
results['FDA'] = acc_fda

In [ ]:
# ============================================================
# Cell 14: Results Summary
# ============================================================
print_results_table(results, 'Art -> Real World')
save_results(results, 'Art_to_RealWorld', save_dir=SAVE_DIR)